In [ ]:
%load_ext autoreload
%autoreload 3

# compression evaluation

In [ ]:
import cartopy.crs as ccrs
import healpix_geo
import healpix_plot
import matplotlib.pyplot as plt
import numpy as np
import xarray as xr
import xdggs  # noqa: F401
import zarr

import hcc  # noqa: F401
import hcc.codecs.dct
import hcc.codecs.grad_dct

## flattening debugging

## generate test data

In [ ]:
level = 17
cell_ids = np.arange(0 * 4**level, 0 * 4**level + 4**9, dtype="uint64")
lon, lat = healpix_geo.nested.healpix_to_lonlat(cell_ids, level, ellipsoid="sphere")
data = np.cos(np.deg2rad(900 * lat)) * np.sin(np.deg2rad(1800 * lon))

ds = xr.Dataset(
    {"data": ("cells", data)}, coords={"cell_ids": ("cells", cell_ids)}
).dggs.decode(
    {
        "grid_name": "healpix",
        "level": 17,
        "indexing_scheme": "nested",
        "ellipsoid": "sphere",
    }
)

## roundtrip with the DCT codec

In [ ]:
dct_store = zarr.storage.MemoryStore()
encoding = {
    "data": {
        "serializer": hcc.codecs.dct.HccDctCodec(delta=2, top_k=14000),
        "chunks": 4**9,
    }
}
ds.dggs.encode(convention="zarr").to_zarr(
    dct_store,
    mode="w",
    encoding=encoding,
    consolidated=False,
)

In [ ]:
grad_dct_store = zarr.storage.MemoryStore()
encoding = {
    "data": {
        "serializer": hcc.codecs.grad_dct.HccGradDctCodec(window=8),
        "chunks": 4**9,
    }
}
ds.dggs.encode(convention="zarr").to_zarr(
    grad_dct_store,
    mode="w",
    encoding=encoding,
    consolidated=False,
)

In [ ]:
uncompressed_store = zarr.storage.MemoryStore()
ds.dggs.encode(convention="zarr").to_zarr(
    uncompressed_store,
    mode="w",
    encoding={"data": {"chunks": 4**9}},
    consolidated=False,
)

## simple evaluation

In [ ]:
def count_objects(store):
    return len(
        [
            name
            for name in store._store_dict
            if name.startswith("data/") and "zarr.json" not in name
        ]
    )


{
    "uncompressed": count_objects(uncompressed_store),
    "dct": count_objects(dct_store),
    "grad_dct": count_objects(grad_dct_store),
}

In [ ]:
def store_size(store):
    sizes = {
        key: value.as_array_like().nbytes for key, value in store._store_dict.items()
    }

    return sum(sizes.values())


{
    "uncompressed": store_size(uncompressed_store) / 1e6,
    "dct": store_size(dct_store) / 1e6,
    "grad_dct": store_size(grad_dct_store) / 1e6,
}

In [ ]:
print(grad_dct_store._store_dict["data/zarr.json"].to_bytes().decode())

In [ ]:
print(dct_store._store_dict["data/zarr.json"].to_bytes().decode())

## value comparison

In [ ]:
reopened_dct = (
    xr.open_dataset(dct_store, engine="zarr", consolidated=False)
    .dggs.decode(convention="zarr")
    .load()
)
reopened_grad_dct = (
    xr.open_dataset(grad_dct_store, engine="zarr", consolidated=False)
    .dggs.decode(convention="zarr")
    .load()
)

reopened_grad_dct

In [ ]:
level = reopened_dct.dggs.grid_info.level

healpix_grid = healpix_plot.HealpixGrid(
    level=level, indexing_scheme="nested", ellipsoid="WGS84"
)

cell_ids = reopened_dct["cell_ids"].values
data_dct = reopened_dct["data"].data

data_grad_dct = reopened_grad_dct["data"].data

In [ ]:
fig, axes = plt.subplots(
    1, 3, subplot_kw={"projection": ccrs.Mollweide()}, figsize=(16, 6)
)

mappable = healpix_plot.plot(
    cell_ids,
    ds["data"].values,
    healpix_grid=healpix_grid,
    sampling_grid={"shape": 512},
    ax=axes[0],
)
axes[0].set_title("original")
fig.colorbar(mappable, orientation="horizontal")

axes[0].coastlines()
axes[0].gridlines(draw_labels="x")
axes[0].gridlines(draw_labels="y")


mappable = healpix_plot.plot(
    cell_ids,
    abs(data_dct - ds.data.values),
    healpix_grid=healpix_grid,
    sampling_grid={"shape": 512},
    ax=axes[1],
    cmap="viridis",
    vmin=0,
    vmax=0.0002,
)
axes[1].set_title("DCT - original")
fig.colorbar(mappable, orientation="horizontal")

axes[1].coastlines()
axes[1].gridlines(draw_labels="x")
axes[1].gridlines(draw_labels="y")

mappable = healpix_plot.plot(
    cell_ids,
    abs(data_grad_dct - ds.data.values),
    healpix_grid=healpix_grid,
    sampling_grid={"shape": 512},
    ax=axes[2],
    cmap="viridis",
    vmin=0,
    vmax=0.0002,
)
axes[2].set_title("GRAD-DCT - original")
fig.colorbar(mappable, orientation="horizontal")

axes[2].coastlines()
axes[2].gridlines(draw_labels="x")
axes[2].gridlines(draw_labels="y")

In [ ]:
hy, hx = np.histogram(data_dct - ds.data.values, bins=10000, range=[-7e-3, 7e-3])
hy2, hx = np.histogram(data_grad_dct - ds.data.values, bins=10000, range=[-7e-3, 7e-3])

l1 = "dct"
l2 = "grad-dct"

plt.plot((hx[1:] + hx[:-1]) / 2, np.sum(hy) - np.cumsum(abs(hy)), color="b", label=l1)
plt.plot((hx[1:] + hx[:-1]) / 2, np.sum(hy) - np.cumsum(abs(hy2)), color="r", label=l2)
plt.yscale("log")
plt.xscale("log")
plt.legend(frameon=0)